# Predição de Violência Municipal com Indicadores Socioeconômicos

**Disciplina:** Aprendizagem de Máquina — UFPB, 2025.1
**Professores:** Bruno Jefferson de Sousa Pessoa · Gilberto Farias de Sousa Filho

---

## Pergunta central
> É possível prever o nível de violência de um município brasileiro usando **apenas indicadores socioeconômicos**?

## Abordagem
Classificação binária: **alta violência** vs **baixa violência**.
Threshold = mediana da taxa de homicídios dolosos per capita (por 100k hab).

## Modelos implementados
- Rede Neural (Keras/TensorFlow)
- Árvore de Decisão (Scikit-Learn)
- SVM — Support Vector Machine (Scikit-Learn)

> ⚠️ **Regra fundamental:** dados de crime são usados **apenas** para construir a variável alvo (y).
> As features são exclusivamente socioeconômicas.

## Imports e configurações

In [ ]:
import os
import re
import warnings
import unicodedata
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve, auc
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

import random
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Tudo carregado!')

---
## 1. Coleta e Preparação dos Dados

Os dados foram obtidos de cinco fontes oficiais:

| Arquivo | Fonte | Variáveis |
|---------|-------|-----------|
| `censo2022_municipios.csv` | IBGE SIDRA — Censo 2022 | população, urbanização, jovens, renda, desemprego, analfabetismo, esgoto |
| `atlas_brasil_municipios.xlsx` | Atlas Brasil — Censo 2010 | IDHM (geral + 3 sub-índices), Gini, % pobres |
| `pib_municipios_sidra_2021.csv` | IBGE SIDRA — Tabela 5938 | PIB municipal → per capita |
| `areas_municipios_2024.xls` | IBGE — Áreas Territoriais | área km² → densidade demográfica |
| `sinesp_municipios.xlsx` | Sinesp/MJ (2018–2022) | homicídios dolosos → variável alvo |

> **Nota sobre anos:** as features do Atlas Brasil são do Censo 2010.
> Não há Gini e IDHM equivalentes publicados a nível municipal no Censo 2022 via SIDRA.
> Esta é uma limitação do projeto discutida no relatório.

### 1.1 Censo 2022 — IBGE SIDRA

In [ ]:
censo = pd.read_csv(RAW_DIR / 'censo2022_municipios.csv', dtype={'cod_ibge': str})
censo['cod_ibge'] = censo['cod_ibge'].str.zfill(7)

print('Shape:', censo.shape)
print('Colunas:', list(censo.columns))
print()
censo.head(3)

### 1.2 Atlas Brasil — Censo 2010

O Atlas Brasil não exporta o código IBGE diretamente — apenas o nome do município no formato `"Nome (UF)"`.
Usamos a API de localidades do IBGE para obter a tabela de referência nome → código IBGE.

In [ ]:
def normalizar(nome):
    """Remove acentos e padroniza para comparação de nomes."""
    if not isinstance(nome, str):
        return ''
    nfkd = unicodedata.normalize('NFKD', nome)
    sem_acento = ''.join(c for c in nfkd if not unicodedata.combining(c))
    return re.sub(r'\s+', ' ', sem_acento).strip().lower()


# Tabela de referência IBGE: nome + UF → código
cache_ibge = RAW_DIR / 'ibge_municipios_ref.csv'
if cache_ibge.exists():
    ibge_ref = pd.read_csv(cache_ibge, dtype={'cod_ibge': str})
    print('Referência IBGE carregada do cache.')
else:
    print('Baixando referência de municípios via API IBGE...')
    r = requests.get(
        'https://servicodados.ibge.gov.br/api/v1/localidades/municipios',
        timeout=60
    )
    dados = r.json()
    rows = []
    for m in dados:
        try:
            uf = m['microrregiao']['mesorregiao']['UF']['sigla']
        except (KeyError, TypeError):
            try:
                uf = m['regiao-imediata']['regiao-intermediaria']['UF']['sigla']
            except (KeyError, TypeError):
                uf = ''
        rows.append({'cod_ibge': str(m['id']), 'nome_norm': normalizar(m['nome']), 'uf': uf})
    ibge_ref = pd.DataFrame(rows)
    ibge_ref.to_csv(cache_ibge, index=False)
    print(f'  {len(ibge_ref)} municípios salvos em cache.')

# Carregar Atlas Brasil
atlas_raw = pd.read_excel(RAW_DIR / 'atlas_brasil_municipios.xlsx')
atlas_raw.columns = ['territorialidade', 'gini', 'idhm', 'idhm_renda',
                     'idhm_longevidade', 'idhm_educacao', 'perc_pobres']

# Filtrar linhas de municípios (formato "Nome (UF)")
atlas_raw = atlas_raw[
    atlas_raw['territorialidade'].str.match(r'^.+\(\w{2}\)$', na=False)
].copy()

# Extrair nome e UF
atlas_raw['nome_norm'] = atlas_raw['territorialidade'].str.extract(r'^(.+)\s+\(\w{2}\)$')[0].apply(normalizar)
atlas_raw['uf']        = atlas_raw['territorialidade'].str.extract(r'\((\w{2})\)$')[0]

# Join com referência IBGE
atlas = atlas_raw.merge(ibge_ref[['cod_ibge', 'nome_norm', 'uf']], on=['nome_norm', 'uf'], how='left')
atlas = atlas[['cod_ibge', 'gini', 'idhm', 'idhm_renda', 'idhm_longevidade', 'idhm_educacao', 'perc_pobres']].copy()

print(f'Atlas Brasil: {len(atlas)} municípios')
print(f'Sem código IBGE (serão tratados na imputação): {atlas["cod_ibge"].isna().sum()}')
atlas.head(3)

### 1.3 PIB per capita — IBGE SIDRA (Tabela 5938, 2021)

PIB per capita = (PIB total em R$) / população do Censo 2022.

In [ ]:
cache_pib = RAW_DIR / 'pib_municipios_sidra_2021.csv'
if cache_pib.exists():
    pib_raw = pd.read_csv(cache_pib, dtype={'cod_ibge': str})
    print('PIB carregado do cache.')
else:
    print('Baixando PIB municipal 2021 via SIDRA API...')
    url = (
        'https://servicodados.ibge.gov.br/api/v3/agregados/5938'
        '/periodos/2021/variaveis/37?localidades=N6[all]'
    )
    r = requests.get(url, timeout=120)
    dados = r.json()
    rows = []
    for bloco in dados[0]['resultados']:
        for item in bloco['series']:
            cod = item['localidade']['id']
            val = item['serie'].get('2021')
            try:
                val = float(val)
            except (TypeError, ValueError):
                val = None
            rows.append({'cod_ibge': cod, 'pib_total_mil': val})
    pib_raw = pd.DataFrame(rows)
    pib_raw.to_csv(cache_pib, index=False)
    print(f'  {len(pib_raw)} municípios salvos.')

# PIB per capita = (PIB em R$ mil × 1.000) / população
pib = pib_raw.merge(censo[['cod_ibge', 'pop_total']], on='cod_ibge', how='left')
pib['pib_per_capita'] = pib['pib_total_mil'] * 1000 / pib['pop_total']
pib = pib[['cod_ibge', 'pib_per_capita']]

print('Shape:', pib.shape)
print('Nulos:', pib['pib_per_capita'].isna().sum())
pib.head(3)

### 1.4 Áreas Territoriais — IBGE 2024

In [ ]:
areas = pd.read_excel(RAW_DIR / 'areas_municipios_2024.xls', dtype={'CD_MUN': str})
areas = areas[['CD_MUN', 'AR_MUN_2024']].rename(
    columns={'CD_MUN': 'cod_ibge', 'AR_MUN_2024': 'area_km2'}
)
print('Shape:', areas.shape)
areas.head(3)

### 1.5 Sinesp — Homicídios Dolosos (2018–2022)

O arquivo do Sinesp possui **uma aba por estado**. Concatenamos todas e filtramos **2022**
(ano mais recente, coincide com o Censo 2022).

> **Decisão de projeto:** municípios sem registro em 2022 recebem **0 homicídios** —
> ausência de registro equivale a ausência de ocorrências.

In [ ]:
xl = pd.ExcelFile(RAW_DIR / 'sinesp_municipios.xlsx')
sinesp = pd.concat([xl.parse(aba) for aba in xl.sheet_names], ignore_index=True)

print('Total de registros (todos os anos):', len(sinesp))
print('Anos disponíveis:', sorted(sinesp['Mês/Ano'].dt.year.unique()))
print('Municípios únicos:', sinesp['Cód_IBGE'].nunique())

# Filtrar 2022 e somar por município
sinesp_2022 = (
    sinesp[sinesp['Mês/Ano'].dt.year == 2022]
    .groupby('Cód_IBGE', as_index=False)['Vítimas'].sum()
    .rename(columns={'Cód_IBGE': 'cod_ibge', 'Vítimas': 'homicidios_2022'})
)
sinesp_2022['cod_ibge'] = sinesp_2022['cod_ibge'].astype(str).str.zfill(7)

print()
print(f'Municípios com registro em 2022: {len(sinesp_2022)}')
sinesp_2022.head(3)

### 1.6 Construindo a variável alvo y

**3 passos:**
1. Calcular a taxa de homicídios por 100k habitantes para cada município
2. Calcular a **mediana nacional** → threshold
3. Binarizar: `alta_violencia = 1` se taxa > mediana, `0` caso contrário

Esta abordagem produz um dataset **perfeitamente balanceado** (50%/50%).

In [ ]:
# Base: todos os municípios do Censo 2022
df_alvo = censo[['cod_ibge', 'pop_total']].merge(sinesp_2022, on='cod_ibge', how='left')
df_alvo['homicidios_2022'] = df_alvo['homicidios_2022'].fillna(0)

# Taxa por 100k
df_alvo['taxa_homicidios'] = df_alvo['homicidios_2022'] / df_alvo['pop_total'] * 100_000

# Threshold = mediana
limiar = df_alvo['taxa_homicidios'].median()
df_alvo['alta_violencia'] = (df_alvo['taxa_homicidios'] > limiar).astype(int)

print(f'Mediana (threshold): {limiar:.2f} homicídios por 100k hab')
print()
print('Distribuição da variável alvo:')
print(df_alvo['alta_violencia'].value_counts())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

counts = df_alvo['alta_violencia'].value_counts().sort_index()
axes[0].bar(['Baixa violência (0)', 'Alta violência (1)'], counts.values,
            color=['steelblue', 'tomato'], edgecolor='white')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, f'{v} ({v/len(df_alvo)*100:.0f}%)', ha='center', fontsize=11)
axes[0].set_title('Distribuição da variável alvo')
axes[0].set_ylabel('Municípios')
axes[0].set_ylim(0, counts.max() * 1.15)

axes[1].hist(df_alvo['taxa_homicidios'], bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(limiar, color='red', linestyle='--', linewidth=2, label=f'Mediana = {limiar:.1f}')
axes[1].set_title('Distribuição da taxa de homicídios')
axes[1].set_xlabel('Homicídios por 100k hab')
axes[1].legend()
plt.tight_layout()
plt.show()

### 1.7 Cruzando todos os datasets

Join pela chave `cod_ibge` (código IBGE do município — 7 dígitos).
Partimos do Censo 2022 (base completa com todos os municípios).
Valores ausentes são imputados pela **mediana** da coluna — máximo 0.6% de nulos.

In [ ]:
FEATURES = [
    'pop_total', 'pop_urbana_pct', 'perc_jovens_15_29', 'renda_per_capita',
    'taxa_desemprego', 'taxa_analfabetismo_15', 'perc_esgoto_adequado',
    'gini', 'idhm', 'idhm_renda', 'idhm_longevidade', 'idhm_educacao',
    'perc_pobres', 'pib_per_capita', 'densidade_demografica'
]

# Início: alvo + pop
df = df_alvo[['cod_ibge', 'pop_total', 'alta_violencia', 'taxa_homicidios']].copy()

# Adicionar features do Censo (exceto pop_total, que já está)
df = df.merge(censo.drop(columns=['pop_total']), on='cod_ibge', how='left')

# Atlas Brasil
df = df.merge(atlas, on='cod_ibge', how='left')

# PIB
df = df.merge(pib, on='cod_ibge', how='left')

# Áreas → densidade demográfica
df = df.merge(areas, on='cod_ibge', how='left')
df['densidade_demografica'] = df['pop_total'] / df['area_km2']

print(f'Shape antes da imputação: {df.shape}')
print()
print('Nulos por feature:')
nulos = df[FEATURES].isnull().sum()
print(nulos[nulos > 0].to_string())

# Imputação pela mediana
for col in FEATURES:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

print()
print(f'Nulos após imputação: {df[FEATURES].isnull().sum().sum()}')

# Salvar dataset final
colunas_finais = ['cod_ibge'] + FEATURES + ['taxa_homicidios', 'alta_violencia']
df[colunas_finais].to_csv(PROCESSED_DIR / 'dataset_municipios.csv', index=False, encoding='utf-8')

print()
print(f'Dataset salvo: {PROCESSED_DIR}/dataset_municipios.csv')
print(f'Shape final:   {df[colunas_finais].shape}')

---
## 2. Análise Exploratória dos Dados

Antes de treinar qualquer modelo, precisamos entender os dados:
distribuições, correlações e relação entre features e o alvo.

In [ ]:
# Recarregar para garantir que trabalhamos com o dataset salvo
df = pd.read_csv(PROCESSED_DIR / 'dataset_municipios.csv')

print(f'Shape: {df.shape}')
print()
df.info()

In [ ]:
print('Estatísticas descritivas das features:')
df[FEATURES].describe().round(2)

In [ ]:
# Histogramas de todas as features
n_cols = 5
n_rows = 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 11))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    axes[i].hist(df[feat], bins=35, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(feat, fontsize=9)
    axes[i].set_ylabel('Freq.', fontsize=8)
    axes[i].tick_params(labelsize=7)

plt.suptitle('Distribuição das 15 features socioeconômicas', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots por classe (baixa vs alta violência)
fig, axes = plt.subplots(3, 5, figsize=(20, 11))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    dados = [df[df['alta_violencia']==0][feat].values,
             df[df['alta_violencia']==1][feat].values]
    bp = axes[i].boxplot(dados, patch_artist=True,
                         labels=['Baixa', 'Alta'],
                         medianprops={'color': 'black', 'linewidth': 2})
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][1].set_facecolor('tomato')
    axes[i].set_title(feat, fontsize=9)
    axes[i].tick_params(labelsize=7)

plt.suptitle('Boxplots das features por classe de violência', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlação
corr = df[FEATURES + ['alta_violencia']].corr()

fig, axes = plt.subplots(1, 2, figsize=(22, 6))

# Heatmap completo
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.4, ax=axes[0], annot_kws={'size': 7})
axes[0].set_title('Matriz de Correlação', fontsize=13)

# Correlação com o alvo
corr_alvo = corr['alta_violencia'].drop('alta_violencia').sort_values()
colors = ['tomato' if v > 0 else 'steelblue' for v in corr_alvo]
corr_alvo.plot(kind='bar', color=colors, ax=axes[1])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlação de cada feature com alta_violencia')
axes[1].set_ylabel('Correlação de Pearson')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Top 5 features mais correlacionadas com violência:')
print(corr_alvo.abs().sort_values(ascending=False).head(5).to_string())

---
## 3. Pré-processamento (seção 4.1 do projeto)

### Construção de X e y
- **X**: matriz com as 15 features socioeconômicas
- **y**: vetor com a variável alvo binária (`alta_violencia`)

### Separação treino/teste — 80%/20%
Com N = 5.570 municípios:
- **Treino (80%)**: ~4.456 amostras — suficiente para cross-validation interna nos 3 modelos
- **Teste (20%)**: ~1.114 amostras — suficiente para estimar E_out com margem de erro ~3%

Usamos `stratify=y` para manter o balanceamento 50/50 em ambos os conjuntos.
**O conjunto de teste não será tocado até a fase de comparação final (seção 7).**

### Padronização — StandardScaler
Transforma cada feature para **média 0 e desvio padrão 1**:

$$x' = \frac{x - \mu}{\sigma}$$

**Por quê StandardScaler?**
- **SVM (kernel RBF):** obrigatório — distâncias no espaço de features dependem da escala
- **Rede Neural:** recomendado — gradientes convergem melhor com entradas centradas
- **Árvore de Decisão:** não afeta os splits, mas aplicamos uniformemente para consistência

**Importante:** o scaler é ajustado (`fit`) **apenas no treino** e aplicado (`transform`) em ambos — evita data leakage.

In [ ]:
X = df[FEATURES].values
y = df['alta_violencia'].values

N, p = X.shape
print(f'N (número de amostras) = {N}')
print(f'p (número de features) = {p}')
print()
print('Features:')
for i, feat in enumerate(FEATURES, 1):
    print(f'  {i:2d}. {feat}')
print()
print(f'Balanceamento: classe 0 = {(y==0).sum()}, classe 1 = {(y==1).sum()}')

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print()
print(f'Treino: {len(X_train)} amostras ({len(X_train)/N*100:.0f}%)')
print(f'Teste:  {len(X_test)} amostras  ({len(X_test)/N*100:.0f}%)')
print()
print(f'Balanceamento treino: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Balanceamento teste:  {dict(zip(*np.unique(y_test,  return_counts=True)))}')

# Padronização
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Validar: médias ≈ 0 e desvios ≈ 1 no treino
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

axes[0].bar(range(p), X_train_s.mean(axis=0), color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Médias após padronização (deve ser ≈ 0)')
axes[0].set_xticks(range(p))
axes[0].set_xticklabels(FEATURES, rotation=45, ha='right', fontsize=7)

axes[1].bar(range(p), X_train_s.std(axis=0), color='tomato')
axes[1].axhline(1, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Desvios padrão após padronização (deve ser ≈ 1)')
axes[1].set_xticks(range(p))
axes[1].set_xticklabels(FEATURES, rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.show()

print('Pré-processamento OK — teste isolado até a seção 7.')

---
## 4. Rede Neural (seção 4.2 do projeto)

### Dimensão VC e Regra de Ouro

A **dimensão VC** de uma rede MLP é aproximada pelo número total de parâmetros (pesos + bias).

**Regra de Ouro:** N ≥ 10 × d_VC

O **Teorema da Aproximação Universal** garante que uma rede com pelo menos 1 camada oculta
e ativação não-linear pode aproximar qualquer função contínua.

### Arquitetura escolhida: [15 → 16 → 8 → 1]
- **2 camadas ocultas** — satisfaz o Teorema da Aproximação Universal
- **ReLU** nas camadas ocultas — evita gradiente desaparecendo
- **Sigmoid** na saída — produz probabilidade em [0, 1]
- **Regularização L2 + Dropout 10%** — previne overfitting leve

### Batch size e épocas
- **Batch size = 64:** equilíbrio entre velocidade e estabilidade do gradiente (~70 batches/época)
- **Máximo 300 épocas + Early Stopping (patience=25):** interrompe quando `val_loss` para de melhorar e restaura os melhores pesos automaticamente

In [ ]:
N_train = X_train_s.shape[0]

# Dimensão VC = soma dos parâmetros de cada camada (neurônios × (entradas + 1 bias))
c1 = (p + 1) * 16    # camada entrada → oculta 1
c2 = (16 + 1) * 8    # oculta 1 → oculta 2
c3 = (8 + 1) * 1     # oculta 2 → saída
d_vc = c1 + c2 + c3

print('=== Dimensão VC — Arquitetura [15 → 16 → 8 → 1] ===')
print()
print(f'Camada entrada → oculta1:  ({p}+1) × 16 = {c1} parâmetros')
print(f'Camada oculta1 → oculta2:  (16+1) × 8  = {c2} parâmetros')
print(f'Camada oculta2 → saída:    (8+1)  × 1  = {c3} parâmetros')
print()
print(f'd_VC total = {d_vc}')
print(f'N treino   = {N_train}')
print(f'Razão N/d_VC = {N_train/d_vc:.1f}')
print()
if N_train >= 10 * d_vc:
    print(f'Regra de Ouro: {N_train} >= 10 × {d_vc} = {10*d_vc}  ✓')
else:
    print('Regra de Ouro NÃO satisfeita — reduzir a arquitetura!')

In [ ]:
# Construir a rede neural
model = Sequential([
    Dense(16, activation='relu', kernel_regularizer=l2(0.0001), input_shape=(p,)),
    Dropout(0.1),
    Dense(8, activation='relu', kernel_regularizer=l2(0.0001)),
    Dropout(0.1),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Separar validação interna do treino (para o Early Stopping)
# O conjunto de TESTE continua isolado
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_s, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f'Treino efetivo:    {len(X_tr)} amostras')
print(f'Validação interna: {len(X_val)} amostras')
print()

early_stop = EarlyStopping(
    monitor='val_loss', patience=25, restore_best_weights=True, verbose=1
)

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=300,
    batch_size=64,
    callbacks=[early_stop],
    verbose=0
)

print()
print(f'Treinamento encerrado na época {len(history.history["loss"])}')

In [ ]:
# Curva de aprendizado — E_in e E_out por época
epocas = range(1, len(history.history['loss']) + 1)
val_loss = history.history['val_loss']
melhor_epoca = val_loss.index(min(val_loss)) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epocas, history.history['loss'],     color='steelblue', label='Loss treino (E_in)')
axes[0].plot(epocas, history.history['val_loss'], color='tomato',    label='Loss validação (E_out)')
axes[0].axvline(melhor_epoca, color='green', linestyle='--', linewidth=1.5,
                label=f'Melhor época: {melhor_epoca}')
axes[0].set_title('Loss por época — Rede Neural')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Binary Crossentropy')
axes[0].legend()

axes[1].plot(epocas, history.history['accuracy'],     color='steelblue', label='Acurácia treino')
axes[1].plot(epocas, history.history['val_accuracy'], color='tomato',    label='Acurácia validação')
axes[1].axvline(melhor_epoca, color='green', linestyle='--', linewidth=1.5,
                label=f'Melhor época: {melhor_epoca}')
axes[1].set_title('Acurácia por época — Rede Neural')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Melhor época (menor val_loss): {melhor_epoca}')
print(f'Se val_loss começa a subir após a época {melhor_epoca}, há overfitting a partir daí.')

In [ ]:
# Predições
y_pred_nn_train = (model.predict(X_train_s, verbose=0) > 0.5).astype(int).ravel()
y_pred_nn       = (model.predict(X_test_s,  verbose=0) > 0.5).astype(int).ravel()

# E_in e E_out
e_in_nn  = 1 - accuracy_score(y_train, y_pred_nn_train)
e_out_nn = 1 - accuracy_score(y_test,  y_pred_nn)
gap_nn   = e_out_nn - e_in_nn

print('=== Rede Neural — Métricas ===')
print(f'E_in  = {e_in_nn:.4f}  ({e_in_nn*100:.2f}%)')
print(f'E_out = {e_out_nn:.4f} ({e_out_nn*100:.2f}%)')
print(f'Gap   = {gap_nn:.4f}  → {"overfitting" if gap_nn > 0.05 else "OK"}')
print()
print(classification_report(y_test, y_pred_nn, target_names=['Baixa violência', 'Alta violência']))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred_nn)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Baixa', 'Alta'], yticklabels=['Baixa', 'Alta'])
plt.title('Matriz de Confusão — Rede Neural')
plt.ylabel('Real')
plt.xlabel('Previsto')
plt.tight_layout()
plt.show()

---
## 5. Árvore de Decisão (seção 4.3 do projeto)

### Estratégia
1. **Treinar sem poda** → demonstrar overfitting (E_in ≈ 0%, E_out alto)
2. **Minimal Cost-Complexity Pruning + Cross Validation (10 folds)** → encontrar o alpha ótimo
   O algoritmo minimiza: Pureza(T) + α × #folhas(T)
3. **Treinar a melhor árvore** com o alpha ótimo e avaliar métricas

O fold de CV é definido por `cv=10` — com N=4.456, cada fold tem ~446 amostras,
proporcionando estimativas com baixo viés.

In [ ]:
# Árvore sem poda — overfitting clássico
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train_s, y_train)

e_in_full  = 1 - accuracy_score(y_train, tree_full.predict(X_train_s))
e_out_full = 1 - accuracy_score(y_test,  tree_full.predict(X_test_s))

print('=== Árvore SEM poda ===')
print(f'Profundidade: {tree_full.get_depth()} | Folhas: {tree_full.get_n_leaves()}')
print(f'E_in  = {e_in_full:.4f} ({e_in_full*100:.2f}%) — memoriza completamente o treino')
print(f'E_out = {e_out_full:.4f} ({e_out_full*100:.2f}%) — generaliza mal')
print(f'Gap   = {e_out_full - e_in_full:.4f} → Overfitting severo!')

# Plot da árvore sem poda (apenas 4 primeiros níveis — a árvore completa é enorme)
fig, ax = plt.subplots(figsize=(28, 10))
plot_tree(
    tree_full, max_depth=4, filled=True,
    feature_names=FEATURES, class_names=['Baixa', 'Alta'],
    ax=ax, fontsize=8, rounded=True
)
ax.set_title(f'Árvore SEM poda — exibindo primeiros 4 de {tree_full.get_depth()} níveis', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Minimal Cost-Complexity: extrair o caminho de alphas possíveis
path = tree_full.cost_complexity_pruning_path(X_train_s, y_train)
alphas = path.ccp_alphas[:-1]   # remover o último alpha (árvore vazia)

print(f'Número de alphas candidatos: {len(alphas)}')
print('Rodando Cross Validation 10-fold para cada alpha...')
print('(pode levar 1-2 minutos)')
print()

cv_mean = []
cv_std  = []
for alpha in alphas:
    tree_tmp = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    scores = cross_val_score(tree_tmp, X_train_s, y_train, cv=10, scoring='f1')
    cv_mean.append(scores.mean())
    cv_std.append(scores.std())

cv_mean = np.array(cv_mean)
cv_std  = np.array(cv_std)

melhor_idx   = np.argmax(cv_mean)
melhor_alpha = alphas[melhor_idx]

print(f'Melhor alpha: {melhor_alpha:.6f}')
print(f'F1 médio (CV 10-fold): {cv_mean[melhor_idx]:.4f} ± {cv_std[melhor_idx]:.4f}')

# Número de folhas por alpha
n_folhas = []
for a in alphas:
    t_tmp = DecisionTreeClassifier(ccp_alpha=a, random_state=42)
    t_tmp.fit(X_train_s, y_train)
    n_folhas.append(t_tmp.get_n_leaves())

# Plot seleção do alpha
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(alphas, cv_mean, color='steelblue', linewidth=1.5)
axes[0].fill_between(alphas, cv_mean - cv_std, cv_mean + cv_std,
                     alpha=0.15, color='steelblue', label='± 1 std')
axes[0].axvline(melhor_alpha, color='red', linestyle='--',
                label=f'α* = {melhor_alpha:.5f}')
axes[0].set_xlabel('Alpha (ccp_alpha)')
axes[0].set_ylabel('F1 médio (CV 10-fold)')
axes[0].set_title('Seleção do alpha — Minimal Cost-Complexity')
axes[0].legend()

axes[1].plot(alphas, n_folhas, color='tomato', linewidth=1.5)
axes[1].axvline(melhor_alpha, color='red', linestyle='--',
                label=f'α* → {n_folhas[melhor_idx]} folhas')
axes[1].set_xlabel('Alpha')
axes[1].set_ylabel('Número de folhas')
axes[1].set_title('Complexidade da árvore vs Alpha')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Treinar a melhor árvore com o alpha ótimo
tree_podada = DecisionTreeClassifier(ccp_alpha=melhor_alpha, random_state=42)
tree_podada.fit(X_train_s, y_train)

y_pred_tree_train = tree_podada.predict(X_train_s)
y_pred_tree       = tree_podada.predict(X_test_s)

e_in_tree  = 1 - accuracy_score(y_train, y_pred_tree_train)
e_out_tree = 1 - accuracy_score(y_test,  y_pred_tree)
gap_tree   = e_out_tree - e_in_tree

print('=== Árvore PODADA ===')
print(f'Alpha: {melhor_alpha:.6f} | Profundidade: {tree_podada.get_depth()} | Folhas: {tree_podada.get_n_leaves()}')
print(f'E_in  = {e_in_tree:.4f}  ({e_in_tree*100:.2f}%)')
print(f'E_out = {e_out_tree:.4f} ({e_out_tree*100:.2f}%)')
print(f'Gap   = {gap_tree:.4f}')
print()
print(classification_report(y_test, y_pred_tree, target_names=['Baixa violência', 'Alta violência']))

# Plot da árvore podada
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    tree_podada, filled=True,
    feature_names=FEATURES, class_names=['Baixa', 'Alta'],
    ax=ax, fontsize=8, rounded=True, impurity=True
)
ax.set_title(
    f'Árvore PODADA — α={melhor_alpha:.5f} | Prof={tree_podada.get_depth()} | Folhas={tree_podada.get_n_leaves()}',
    fontsize=13
)
plt.tight_layout()
plt.show()

# Importância das features
imp = tree_podada.feature_importances_
idx = np.argsort(imp)[::-1]

plt.figure(figsize=(10, 4))
plt.bar(range(p), imp[idx], color='steelblue', edgecolor='white')
plt.xticks(range(p), [FEATURES[i] for i in idx], rotation=45, ha='right', fontsize=8)
plt.title('Importância das features — Árvore de Decisão (Gini)')
plt.ylabel('Importância')
plt.tight_layout()
plt.show()

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred_tree)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Baixa', 'Alta'], yticklabels=['Baixa', 'Alta'])
plt.title('Matriz de Confusão — Árvore de Decisão')
plt.ylabel('Real')
plt.xlabel('Previsto')
plt.tight_layout()
plt.show()

---
## 6. SVM — Support Vector Machine (seção 4.4 do projeto)

Usamos o kernel **RBF** (Radial Basis Function): K(x, x') = exp(−γ ‖x − x'‖²)

### Hiperparâmetros
- **C:** penaliza erros de classificação. Alto → margem estreita (risco overfitting); Baixo → margem larga (risco underfitting)
- **γ (gamma):** raio de influência de cada amostra. Alto → muito local (overfitting); Baixo → muito global (underfitting)

### GridSearchCV com 10-fold CV
Testamos 5 × 5 = 25 combinações. `scoring='f1'` para consistência com os outros modelos.

### E_out esperado — Teorema LOO
Pelo teorema Leave-One-Out do SVM:

E_out ≤ (número de vetores de suporte) / N_treino

In [ ]:
param_grid = {
    'C':     [0.01, 0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 'scale'],
}

print('Buscando melhores hiperparâmetros com GridSearchCV (10-fold CV)...')
print('Isso pode levar alguns minutos...')

svm_model = SVC(kernel='rbf', random_state=42)
grid = GridSearchCV(svm_model, param_grid, cv=10, scoring='f1', n_jobs=-1, verbose=0)
grid.fit(X_train_s, y_train)

print()
print(f'Melhor C:     {grid.best_params_["C"]}')
print(f'Melhor gamma: {grid.best_params_["gamma"]}')
print(f'Melhor F1 (CV 10-fold): {grid.best_score_:.4f}')

# Heatmap dos resultados
res = pd.DataFrame(grid.cv_results_)
res['gamma_str'] = res['param_gamma'].astype(str)
pivot = res.pivot_table(index='param_C', columns='gamma_str', values='mean_test_score')

plt.figure(figsize=(10, 5))
sns.heatmap(pivot.astype(float), annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5)
plt.title('GridSearchCV — F1 médio (CV 10-fold) por C e gamma')
plt.ylabel('C')
plt.xlabel('gamma')
plt.tight_layout()
plt.show()

In [ ]:
best_svm = grid.best_estimator_

y_pred_svm_train = best_svm.predict(X_train_s)
y_pred_svm       = best_svm.predict(X_test_s)

e_in_svm  = 1 - accuracy_score(y_train, y_pred_svm_train)
e_out_svm = 1 - accuracy_score(y_test,  y_pred_svm)
gap_svm   = e_out_svm - e_in_svm

# E_out esperado — Teorema LOO
n_sv = best_svm.n_support_.sum()
e_out_esp = n_sv / len(X_train_s)

print(f'=== SVM — C={grid.best_params_["C"]}, gamma={grid.best_params_["gamma"]} ===')
print()
print(f'Vetores de suporte: {n_sv} de {len(X_train_s)} ({e_out_esp*100:.1f}% do treino)')
print(f'  Classe 0 (Baixa): {best_svm.n_support_[0]}')
print(f'  Classe 1 (Alta):  {best_svm.n_support_[1]}')
print()
print(f'E_in               = {e_in_svm:.4f}  ({e_in_svm*100:.2f}%)')
print(f'E_out (empírico)   = {e_out_svm:.4f} ({e_out_svm*100:.2f}%)')
print(f'E_out (esperado ≤) = {e_out_esp:.4f} ({e_out_esp*100:.2f}%)')
print(f'Gap                = {gap_svm:.4f}')
print()
if e_out_svm <= e_out_esp:
    print('E_out empírico < E_out esperado ✓  — limitante teórico respeitado')
else:
    print('Atenção: E_out empírico > E_out esperado')
print()
print(classification_report(y_test, y_pred_svm, target_names=['Baixa violência', 'Alta violência']))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Baixa', 'Alta'], yticklabels=['Baixa', 'Alta'])
plt.title('Matriz de Confusão — SVM')
plt.ylabel('Real')
plt.xlabel('Previsto')
plt.tight_layout()
plt.show()

---
## 7. Comparação dos Modelos e Escolha do Melhor (seção 4.5 do projeto)

Primeiro e único uso do conjunto de **teste** para Árvore e SVM (para a rede neural o teste já
foi usado na seção 4 para monitorar overfitting — aqui consolidamos tudo).

**Critério de escolha:** maior **F1-score** no conjunto de teste.
O F1 é mais informativo que a acurácia para classificação binária — captura o equilíbrio
entre precisão (não alarmar falsos positivos) e recall (não perder municípios de alto risco).

In [ ]:
# Tabela comparativa
tabela = pd.DataFrame({
    'Rede Neural': {
        'Acurácia': accuracy_score(y_test, y_pred_nn),
        'Precisão': precision_score(y_test, y_pred_nn),
        'Recall':   recall_score(y_test, y_pred_nn),
        'F1-score': f1_score(y_test, y_pred_nn),
        'E_in':     e_in_nn,
        'E_out':    e_out_nn,
        'Gap':      gap_nn,
    },
    'Árvore de Decisão': {
        'Acurácia': accuracy_score(y_test, y_pred_tree),
        'Precisão': precision_score(y_test, y_pred_tree),
        'Recall':   recall_score(y_test, y_pred_tree),
        'F1-score': f1_score(y_test, y_pred_tree),
        'E_in':     e_in_tree,
        'E_out':    e_out_tree,
        'Gap':      gap_tree,
    },
    'SVM': {
        'Acurácia': accuracy_score(y_test, y_pred_svm),
        'Precisão': precision_score(y_test, y_pred_svm),
        'Recall':   recall_score(y_test, y_pred_svm),
        'F1-score': f1_score(y_test, y_pred_svm),
        'E_in':     e_in_svm,
        'E_out':    e_out_svm,
        'Gap':      gap_svm,
    }
}).T.round(4)

print('=== Tabela Comparativa — Conjunto de Teste (N=1114) ===')
print()
print(tabela.to_string())

# Gráfico de métricas
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

tabela[['Acurácia', 'Precisão', 'Recall', 'F1-score']].plot(
    kind='bar', ax=axes[0], edgecolor='white'
)
axes[0].set_ylim(0.5, 0.85)
axes[0].set_title('Métricas de qualidade por modelo')
axes[0].set_xticklabels(tabela.index, rotation=0)
axes[0].legend(loc='lower right')

tabela[['E_in', 'E_out', 'Gap']].plot(
    kind='bar', ax=axes[1], color=['steelblue', 'tomato', 'gray'], edgecolor='white'
)
axes[1].set_title('E_in, E_out e Gap por modelo')
axes[1].set_xticklabels(tabela.index, rotation=0)
axes[1].set_ylabel('Erro')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC — todos os modelos
score_nn   = model.predict(X_test_s, verbose=0).ravel()
score_tree = tree_podada.predict_proba(X_test_s)[:, 1]
score_svm  = best_svm.decision_function(X_test_s)

plt.figure(figsize=(8, 7))

for nome, scores, cor in [
    ('Rede Neural',       score_nn,   'steelblue'),
    ('Árvore de Decisão', score_tree, 'tomato'),
    ('SVM',               score_svm,  'seagreen'),
]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=cor, lw=2, label=f'{nome} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Classificador aleatório (AUC = 0.500)')
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos (Recall)')
plt.title('Curvas ROC — Comparação dos 3 modelos')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Decisão final
melhor_modelo = tabela['F1-score'].idxmax()

print('=' * 50)
print(f'MELHOR MODELO: {melhor_modelo}')
print('=' * 50)
print()
print(f'F1-score: {tabela.loc[melhor_modelo, "F1-score"]:.4f}')
print(f'Acurácia: {tabela.loc[melhor_modelo, "Acurácia"]:.4f}')
print(f'E_out:    {tabela.loc[melhor_modelo, "E_out"]:.4f}')
print(f'Gap:      {tabela.loc[melhor_modelo, "Gap"]:.4f}')
print()
print('Justificativa:')
print('  - Maior F1-score: melhor equilíbrio entre precisão e recall')
print('  - Menor gap E_out-E_in: melhor generalização')
print('  - Maior AUC-ROC: melhor poder discriminativo em todos os thresholds')

---
## Conclusão

### Resposta à pergunta central
> **"É possível prever o nível de violência de um município usando apenas indicadores socioeconômicos?"**

**Sim, com desempenho moderado e acima do acaso.** O melhor modelo acerta ~66% dos municípios
— 16 pontos percentuais acima de um chute aleatório (50%). A AUC-ROC > 0.72 confirma
capacidade discriminativa real em todos os limiares de decisão.

### Features mais importantes
- `perc_jovens_15_29` — maior correlação positiva com violência
- `idhm_longevidade`, `renda_per_capita`, `gini` — desenvolvimento e desigualdade
- `taxa_analfabetismo_15`, `idhm_educacao` — educação como fator protetor

### Limitações
1. Features do Atlas Brasil são do **Censo 2010** — defasagem de 12 anos em relação ao alvo de 2022
2. Dados do Sinesp com subnotificação em municípios pequenos
3. Classificação na mediana cria ambiguidade em municípios próximos do threshold
4. Fatores importantes ausentes: presença de organizações criminosas, políticas de segurança, etc.